In [ ]:
!pip install hyperas
!pip install hyperopt

     |████████████████████████████████| 1.1MB 3.8MB/s 
  Found existing installation: pyzmq 17.0.0
    Uninstalling pyzmq-17.0.0:
      Successfully uninstalled pyzmq-17.0.0


In [ ]:
import os

# os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
# from sklearn.model_selection import RandomizedSearchCV
# from tensorflow_core.python.keras.wrappers.scikit_learn import KerasClassifier

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '1'

import tensorflow as tf
import numpy as np
import random as rn

sd = 0  # Here sd means seed.
np.random.seed(1)
rn.seed(2)

config = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
tf.compat.v1.set_random_seed(3)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=config)

# # from keras import backend as K
# NUM_PARALLEL_EXEC_UNITS = 4
# config = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=NUM_PARALLEL_EXEC_UNITS, inter_op_parallelism_threads=2,
#                        allow_soft_placement=True, device_count={'CPU': NUM_PARALLEL_EXEC_UNITS})
# session = tf.compat.v1.Session(config=config)
# tf.compat.v1.keras.backend.set_session(session)
# os.environ["OMP_NUM_THREADS"] = "4"
# os.environ["KMP_BLOCKTIME"] = "30"
# os.environ["KMP_SETTINGS"] = "1"
# os.environ["KMP_AFFINITY"] = "granularity=fine,verbose,compact,1,0"
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [ ]:
import matplotlib.pyplot as plt
import gensim
import pandas as pd
import tensorflow.keras
import tensorflow.keras.layers
import nltk
import sklearn
from __future__ import print_function
from hyperopt import Trials, STATUS_OK, tpe
from hyperas import optim
from hyperas.distributions import choice, uniform
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation
from keras.datasets import mnist
from keras.utils import np_utils
import re

Using TensorFlow backend.


In [ ]:
def data():
    '''
    Data providing function:
    This function is separated from model() so that hyperopt
    won't reload data for each evaluation run.
    '''




    def reset_seeds(reset_graph_with_backend=None):
        if reset_graph_with_backend is not None:
            K = reset_graph_with_backend
            K.clear_session()
            tf.compat.v1.reset_default_graph()
            print("KERAS AND TENSORFLOW GRAPHS RESET")  # optional

            np.random.seed(1)
            rn.seed(2)
            tf.compat.v1.set_random_seed(3)
            print("RANDOM SEEDS RESET")  # optional


    reset_seeds()
    file_location = 'C:/Users/asus/PycharmProjects/TESIS2019/CSV file/Labeled data/'
    file_location_test = 'C:/Users/asus/PycharmProjects/TESIS2019/CSV file/Labeled data/'
    train_data_name = 'jokowimundurlahALL_Balanced'
    test_data_name = 'jokowimundurlahB2(8001-8418)'
    data = pd.read_csv(file_location + train_data_name + '.csv', sep=';', encoding='latin-1')
    data_test = pd.read_csv(file_location_test + test_data_name + '.csv', sep=';', encoding='latin-1')

    print(data.head())

    print(data_test.head())

    print(data.shape)

    print(data_test.head())

    print(data.Label.value_counts())




    def remove_punct(text):
        # text_nopunct = ''
        text_nopunct = re.sub('[' + '^A-Za-z0-9@_.' + ']', ' ', text)
        return text_nopunct


    data['Comment_Clean'] = data['Comment'].apply(lambda x: remove_punct(x))
    print(data['Comment_Clean'])

    tokens = [nltk.word_tokenize(sen) for sen in data['Comment_Clean']]

    def lower_token(tokens):
        return [w.lower() for w in tokens]


    lower_tokens = [lower_token(token) for token in tokens]
    # print(lower_tokens)

    stoplist = (
        open('C:/Users/asus/PycharmProjects/TESIS2019/Word2vecCNN/tala-stopword.txt', 'r').read().replace('\n', ' ')).split(
        " ")


    def removeStopWords(tokens):
        return [word for word in tokens if word not in stoplist]


    filtered_words = [removeStopWords(sen) for sen in lower_tokens]
    # filtered_words = lower_tokens
    data['Comment_Final'] = [' '.join(sen) for sen in filtered_words]
    data['tokens'] = filtered_words
    # print(data['Comment_Final'])
    # print(data['tokens'])

    Not_Hate = []
    Hate = []
    for l in data['Label']:
        if l == 'Not_Hate':
            Not_Hate.append(1)
            Hate.append(0)
            # neg.append(1)
        elif l == 'Hate':
            Not_Hate.append(0)
            # pos.append(1)
            Hate.append(1)

    data['Not_Hate'] = Not_Hate
    print(data['Not_Hate'])
    data['Hate'] = Hate
    print(data['Hate'])

    data = data[['Comment_Final', 'tokens', 'Label', 'Not_Hate', 'Hate']]
    data.head()
    print(data.head())

    all_training_words = [word for tokens in data['tokens'] for word in tokens]
    training_sentence_lengths = [len(tokens) for tokens in data['tokens']]
    TRAINING_VOCAB = sorted(list(set(all_training_words)))
    print("%s words total, with a vocabulary size of %s" % (len(all_training_words), len(TRAINING_VOCAB)))
    print("Max sentence length is %s" % max(training_sentence_lengths))

    # # MODEL WORD2VEC

    def hash(astring):
        return ord(astring[0])


    EMBEDDING_DIM = 100  # best Conv1DNet2 = 200, Conv1DNetKeras = 200,100, 300
    # sample = 1e-3
    # # best iter=30 windows=30

    #CHANGE W2V
    num_epochs = 10
    batch_size = 32
    namew2v = 'C:/Users/asus/PycharmProjects/TESIS2019/Word2vecCNN/RunW2VCNN/w2vbaruakurat/Word2Vec15windowbalanced'
    # w2v_model = gensim.models.Word2Vec(data['tokens'], size=EMBEDDING_DIM, min_count=1, iter=1000, sg=1, window=15,
    #                                    workers=1, hashfxn=hash, seed=sd)
    # print(w2v_model[data['tokens'][0][0]])
    # w2v_model.save('./'+namew2v+'.bin')
    # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    word2vec_path = namew2v+'.bin'
    word2vec = gensim.models.KeyedVectors.load(word2vec_path)
    print(word2vec)


    def get_average_word2vec(tokens_list, vector, generate_missing=False, k=EMBEDDING_DIM):
        if len(tokens_list) < 1:
            return np.zeros(k)
        if generate_missing:
            vectorized = [vector[word] if word in vector else np.random.rand(k) for word in tokens_list]
        else:
            vectorized = [vector[word] if word in vector else np.zeros(k) for word in tokens_list]
        length = len(vectorized)
        summed = np.sum(vectorized, axis=0)
        averaged = np.divide(summed, length)
        return averaged


    def get_word2vec_embeddings(vectors, clean_comments, generate_missing=False):
        embeddings = clean_comments['tokens'].apply(
            lambda x: get_average_word2vec(x, vectors, generate_missing=generate_missing))
        return list(embeddings)


    training_embeddings = get_word2vec_embeddings(word2vec, data, generate_missing=True)

    MAX_SEQUENCE_LENGTH = max(training_sentence_lengths)  # DEFAULT NUMBER 0..253 etc
    tokenizer = tensorflow.keras.preprocessing.text.Tokenizer(num_words=len(TRAINING_VOCAB), lower=True, char_level=False)
    tokenizer.fit_on_texts(data["Comment_Final"].tolist())
    training_sequences = tokenizer.texts_to_sequences(data["Comment_Final"].tolist())

    train_word_index = tokenizer.word_index
    print('Found %s unique tokens.' % len(train_word_index))

    train_cnn_data = tensorflow.keras.preprocessing.sequence.pad_sequences(training_sequences, maxlen=MAX_SEQUENCE_LENGTH)

    train_embedding_weights = np.zeros((len(train_word_index) + 1, EMBEDDING_DIM))
    for word, index in train_word_index.items():
        train_embedding_weights[index, :] = word2vec[word] if word in word2vec else np.random.rand(EMBEDDING_DIM)
    print(train_embedding_weights.shape)


    label_names = ['Not_Hate', 'Hate']

    X = train_cnn_data
    y = data[label_names].values

    X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(X, y, test_size=0.2, stratify=y,
                                                                              random_state=sd)
    return X_train, y_train, X_val, y_val

In [ ]:
def model(X_train, Y_train, X_test, Y_test, sd, ):
    '''
    Model providing function:
    Create Keras model with double curly brackets dropped-in as needed.
    Return value has to be a valid python dictionary with two customary keys:
        - loss: Specify a numeric evaluation metric to be minimized
        - status: Just use STATUS_OK and see hyperopt documentation if not feasible
    The last one is optional, though recommended, namely:
        - model: specify the model just created so that we can later use it again.
    '''
    reset_seeds()
    embedding_layer = tensorflow.keras.layers.Embedding(num_words,
                                             embedding_dim,
                                             weights=[embeddings],
                                             input_length=max_sequence_length,
                                             trainable=False)

    sequence_input = tensorflow.keras.Input(shape=(max_sequence_length,), dtype='int32')
    embedded_sequences = embedding_layer(sequence_input)

    convs = []
    filter_sizes = [2, 3, 4, 5, 6]
    # num_filters = 200  # default 200
    init = tensorflow.keras.initializers.random_normal(seed=sd)
    for filter_size in filter_sizes:
        l_conv = tensorflow.keras.layers.Conv1D(filters=num_filters, kernel_size=filter_size, activation='relu')(
            embedded_sequences)
        l_pool = tensorflow.keras.layers.GlobalMaxPooling1D()(l_conv)
        convs.append(l_pool)

    l_merge = tensorflow.keras.layers.concatenate(convs, axis=1)
    drops1 = tensorflow.keras.layers.Dropout(drop1, seed=sd)(l_merge)
    dense = tensorflow.keras.layers.Dense(dense, activation='relu')(drops1)  # default 128
    drops2 = tensorflow.keras.layers.Dropout(drop2, seed=sd)(dense)  # default 0.2
    preds = tensorflow.keras.layers.Dense(labels_index, activation='sigmoid')(drops2)  # default sigmoid

    model = tensorflow.keras.models.Model(sequence_input, preds)
    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['acc'])
    model.summary()

    model.fit(X_train, Y_train,
              batch_size={{choice([8, 16, 32, 64, 128])}},
              nb_epoch=10,
              verbose=2,
              validation_data=(X_val, y_val))
    score, acc = model.evaluate(X_val, y_val, verbose=0)
    print('Test accuracy:', acc)
    return {'loss': -acc, 'status': STATUS_OK, 'model': model}

In [ ]:
# See: https://stackoverflow.com/questions/49920031/get-the-path-of-the-notebook-on-google-colab
# Install the PyDrive wrapper & import libraries.
!pip install -U -q PyDrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Copy/download the file
fid = drive.ListFile({'q':"title='Untitled4.ipynb'"}).GetList()[0]['id']
f = drive.CreateFile({'id': fid})
f.GetContentFile('Untitled4.ipynb')

In [ ]:
best_run, best_model = optim.minimize(model=model,
                                          data=data,
                                          max_evals=10,
                                          algo=tpe.suggest,
                                          notebook_name='Untitled4', # This is important!
                                          trials=Trials())